In [46]:
import pandas as pd
import numpy as np
import gzip
import matplotlib.pyplot as plt
import pickle
from scipy.stats import linregress
import os
import sys
from datetime import date
from sklearn.metrics import auc
import random
from sklearn.decomposition import NMF, non_negative_factorization

In [43]:
DHS_metadata = pd.read_table('/home/ctrader/Hold/DHS/DHS_Index_and_Vocabulary_hg38_WM20190703.txt.gz', header=None)

col_names = DHS_metadata.iloc[0]
DHS_metadata = DHS_metadata[1:]
DHS_metadata.columns = col_names
DHS_metadata = DHS_metadata.reset_index()
DHS_metadata[['identifier']] = DHS_metadata[['identifier']].astype(str)
DHS_metadata = DHS_metadata.drop(columns=['index'])

tsv_file = open("/home/ctrader/Hold/DHS/DHS_Index_and_Vocabulary_metadata.tsv")
df = pd.read_csv(tsv_file, delimiter="\t")
biosample_df = df[['library order', 'Biosample name']]

/home/ctrader/.conda/envs/lightfm/lib/python3.8/site-packages/IPython/core/interactiveshell.py:3441: DtypeWarning: Columns (1,2,3,4,5,6,7,8) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [34]:
#reordering dataframe to match order of presence/absence matrix and DHS metadata dataframe
#first bigwig file
# GM06990_signal = pd.read_table('/home/ctrader/lightfm_data/out_ENCFF691MRD.txt', names=['name', 'size', 'covered', 'sum', 'mean0', 'mean'])

#second bigwig file
GM06990_signal = pd.read_table('/home/ctrader/lightfm_data/out_ENCFF962FBP.txt', names=['name', 'size', 'covered', 'sum', 'mean0', 'mean'])

GM06990_signal[['name']] = GM06990_signal[['name']].astype(str)
GM06990_signal = GM06990_signal.set_index('name')
id_order = DHS_metadata['identifier'].to_list()
GM06990_signal = GM06990_signal.reindex(id_order)
GM06990_signal = GM06990_signal.reset_index()

In [2]:
#read in 733 sample x 3.5+ million DHS presence/absence matrix
dhs = pd.read_table('/home/ctrader/Hold/DHS/dat_bin_FDR01_hg38.txt.gz', header=None).T
dhs.shape

(733, 3591898)

In [3]:
#Loading biosample embeddings
fn = gzip.GzipFile('/home/ctrader/Hold/DHS/2018-06-08NC16_NNDSVD_Basis.npy.gz', "r")
decomp_Basis = np.load(fn)

In [4]:
#Loading DHS embeddings
fn = gzip.GzipFile('/home/ctrader/Hold/DHS/2018-06-08NC16_NNDSVD_Mixture.npy.gz', "r")
decomp_Mixture = np.load(fn)

In [13]:
decomp_Mixture.shape

(16, 3591898)

In [5]:
print(decomp_Basis.shape)
print(decomp_Mixture.shape)

(733, 16)
(16, 3591898)


In [6]:
dhs_array = dhs.to_numpy()

In [7]:
n_biosamples, n_biosample_e = decomp_Basis.shape
n_DHS_e, n_DHSs = decomp_Mixture.shape
print(n_biosamples, n_DHSs)

733 3591898


**Fill in DHS matrix**

In [8]:
#Generate scores for each interaction
scores = np.dot(decomp_Basis, decomp_Mixture)
scores.shape

(733, 3591898)

In [9]:
#finding known positives for all biosamples
full_kp = np.where(dhs_array==1)

In [10]:
#sorting scores for each biosample
sorted_scores = np.argsort(scores, axis=1)

In [11]:
cut_size = 1000

In [12]:
#for each row, updating DHS entries based on top predicted scores (ie top cut_size DHS indices=1)
updated_dhs_array = dhs_array
for biosample in range(n_biosamples):
    filtered_sorted_scores = np.delete(sorted_scores[biosample], np.where(np.isin(sorted_scores[biosample], full_kp[1][full_kp[0]==biosample])))
    first_cut_dhss = filtered_sorted_scores[:cut_size]
    updated_dhs_array[biosample, first_cut_dhss] = 1

In [37]:
new_dhs = pd.DataFrame(updated_dhs_array)
new_dhs.shape

(733, 3591898)

In [27]:
#regenerate NMF decompositions
new_decomp_Basis, new_decomp_Mixture, n_iter = non_negative_factorization(updated_dhs_array, n_components=16, init='nndsvd', random_state=3, verbose=1)

violation: 1.0
violation: 0.4071233452679745
violation: 0.23853663666470412
violation: 0.1607055895698334
violation: 0.12611537818916527
violation: 0.10902432234448053
violation: 0.09743116680069404
violation: 0.0883046657804241
violation: 0.07988707023352991
violation: 0.07245562984263902
violation: 0.06596866227751731
violation: 0.05984096842153321
violation: 0.05336926246930665
violation: 0.046536469052453004
violation: 0.039823507735249224
violation: 0.03392895869834535
violation: 0.029117497403331954
violation: 0.025355420729204364
violation: 0.022477670088976907
violation: 0.02030434973127502
violation: 0.018692431231669793
violation: 0.017441424224555375
violation: 0.01648423613947482
violation: 0.01573016614936573
violation: 0.01509365301572973
violation: 0.014555591477914975
violation: 0.014106474788602524
violation: 0.013725841139661002
violation: 0.013391434604432593
violation: 0.013034490168611055
violation: 0.012674151371064906
violation: 0.012294488210724937
violation: 0.

In [51]:
new_dhs, updated_dhs_array, new_decomp_Basis, new_decomp_Mixture = fill_in(cut_size=1000, cycles=3)

Starting Cycle 0 / 3
violation: 1.0
violation: 0.4044467507051586
violation: 0.21506103474052887
violation: 0.1444591450755274
violation: 0.11703955531073058
violation: 0.10090184788513519
violation: 0.08719893365833659
violation: 0.0757681908042615
violation: 0.06744731219870728
violation: 0.06179029985572319
violation: 0.05790690442483115
violation: 0.055098631414608135
violation: 0.052240119524358315
violation: 0.04846858087749596
violation: 0.04369174313510249
violation: 0.03813658697503254
violation: 0.032728429156910065
violation: 0.028074766177690672
violation: 0.02435495899872635
violation: 0.021433971446515508
violation: 0.019127634162336673
violation: 0.017327332666060707
violation: 0.015929042087947643
violation: 0.01480675219756313
violation: 0.013883496629229755
violation: 0.013168272980257936
violation: 0.012592463798009426
violation: 0.01210076361095626
violation: 0.011685705625160919
violation: 0.011330506373101162
violation: 0.011065054554162965
violation: 0.0108574086

violation: 0.00020594920618962392
violation: 0.00020186066551050778
violation: 0.0001942496438598528
violation: 0.0001858066578301825
violation: 0.00017704082282301968
violation: 0.0001684007628092871
violation: 0.00016005478541560408
violation: 0.00015203592269836207
violation: 0.00014400382839778277
violation: 0.0001360400089760944
violation: 0.00012858036822749157
violation: 0.00012146764605826898
violation: 0.0001146762408959895
violation: 0.00010820993778495182
violation: 0.00010207283076836585
violation: 9.626320166424175e-05
Converged at iteration 133
check
Starting Cycle 2 / 3
violation: 1.0
violation: 0.39956312832733837
violation: 0.21711100513247591
violation: 0.1460120677688035
violation: 0.11046619943188596
violation: 0.08730965174801232
violation: 0.07222077133026461
violation: 0.06380658046985961
violation: 0.060209380871242
violation: 0.05899078190888176
violation: 0.058110077997604495
violation: 0.05677328312648342
violation: 0.05432180693393065
violation: 0.0500789693

TypeError: cannot unpack non-iterable NoneType object

In [50]:
def fill_in(Basis=decomp_Basis, Mixture=decomp_Mixture, dhs_arr=dhs_array, cut_size=1000, cycles=1):
    n_biosamples, n_biosample_e = Basis.shape
    n_DHS_e, n_DHSs = Mixture.shape
    for i in range(cycles):
        print('Starting Cycle %s / %s' %(i, cycles))
        #finding known positives for all biosamples
        full_kp = np.where(dhs_arr==1)
        #Generate scores for each interaction
        scores = np.dot(Basis, Mixture)
        #sorting scores for each biosample
        sorted_scores = np.argsort(scores, axis=1)
        updated_dhs_array = dhs_arr
        #for each row, updating DHS entries based on top predicted scores (ie top cut_size DHS indices=1)
        for biosample in range(n_biosamples):
            filtered_sorted_scores = np.delete(sorted_scores[biosample], np.where(np.isin(sorted_scores[biosample], full_kp[1][full_kp[0]==biosample])))
            first_cut_dhss = filtered_sorted_scores[:cut_size]
            updated_dhs_array[biosample, first_cut_dhss] = 1
        #regenerate NMF decompositions
        new_decomp_Basis, new_decomp_Mixture, n_iter = non_negative_factorization(updated_dhs_array, n_components=16, init='nndsvd', random_state=3, verbose=1)
        #for new cycles
        if i != cycles:
            print('check')
            dhs_arr = updated_dhs_array
            Basis = new_decomp_Basis
            Mixture = new_decomp_Mixture
        else:
            new_dhs = pd.DataFrame(updated_dhs_array)
            return new_dhs, updated_dhs_array, new_decomp_Basis, new_decomp_Mixture

In [45]:
rec_info = sample_recommendation([0], GM06990_signal, print_num=15, biosample_e=new_decomp_Basis, DHS_e=new_decomp_Mixture, dhs=new_dhs, ROIurl=False)

Biosample: GM06990 (0)
   Number of Known DHSs: 83918

   Recommended DHS Locations: 
   Location | Component | Number of Samples | Biosample Signal/Average Signal | Left +DHS Signal | Right +DHS Signal
      chr11:110201459-110201620 | Tissue invariant | 688 | 0.877/2.833 | N/A | N/A
      chr17:8410360-8410560 | Tissue invariant | 707 | 0.813/3.276 | N/A | N/A
      chr1:248906428-248906799 | Tissue invariant | 713 | 4.982/4.91 | 12.514/4.274 | N/A
      chr5:150561043-150561202 | Tissue invariant | 654 | 0.909/3.34 | N/A | N/A
      chr21:32771960-32772200 | Tissue invariant | 732 | 4.836/9.735 | 12.623/7.252 | N/A
      chr19:57435140-57435360 | Tissue invariant | 732 | 2.896/9.453 | N/A | N/A
      chr14:22766540-22766740 | Tissue invariant | 728 | 1.91/8.802 | N/A | 2.646/1.699
      chr1:629870-630020 | Tissue invariant | 730 | 0.0/146.867 | N/A | N/A
      chrX:119943674-119943880 | Tissue invariant | 728 | 2.577/4.075 | 1.383/1.174 | N/A
      chr1:173714820-173715060 | Tissue

**Functions needed to generate recommendations**

In [31]:
#function to recommend DHSs for a given biosample. Takes in the model, data, and a list of
#biosamples for DHSs to be recommend for. For each biosample, prints out known associated DHSs
#and a list of recommended DHSs. Function modeled after tutorial 3, 'Recommendation Systems
#- Learn Python for Data Science' on LightFM github. 

def sample_recommendation(biosamples, signal_data, print_num=3, biosample_e=decomp_Basis, DHS_e=decomp_Mixture, dhs=dhs, ROIurl=False, comps_to_filter_out=[]): #, full_data=gd_coo
    
    #number of biosamples and DHSs in dataset
    n_biosamples, n_biosample_e = biosample_e.shape
    n_DHS_e, n_DHSs = DHS_e.shape
    return_list = []
    metadata = DHS_metadata.reset_index()
    comp_ser = metadata['component']
    chrom_ser = metadata['seqname']
    start_ser = metadata['start']
    end_ser = metadata['end']
    numsamples_ser = metadata['numsamples']
    identifier_ser = metadata['identifier']
    
    #generate DHS recommendations for each biosample
    for biosample_name in biosamples:
        if isinstance(biosample_name, str):
            biosample = list(biosample_df[biosample_df['Biosample name']==biosample_name]['library order'])[0] - 1
            
        else:
            biosample = biosample_name
            if (biosample > 732) & (n_biosamples == 747): #Pancreatic Cancer data
                relative_num = biosample - 733
                biosample_name = pc_sample_names[relative_num]
            elif (biosample > 732) & (n_biosamples == 771): #Immune data
                relative_num = biosample - 733
                biosample_name = immune_sample_names[relative_num]
            else:
                biosample_name = list(biosample_df[biosample_df['library order']==biosample+1]['Biosample name'])[0]
            
        
        #DHSs known to relate to the biosample
        #indices
        known_positives = [i for i, dhs_pos in enumerate(list(dhs.T[biosample])) if dhs_pos == 1]
        
        #filter out known positives from what will be recommended
        not_known_ind = list(set(np.arange(n_DHSs)) - set(known_positives))
        
        nki_series = pd.Series(not_known_ind)
        
        #DHSs model predictions that relate to biosample
        filtered_scores = np.dot(biosample_e[biosample, :], DHS_e[:, np.array(not_known_ind)])
        

        #rank from most related to least
        #not DHS indices
        filtered_top_DHSs = [np.argsort(-filtered_scores)][0]
        top_rec_DHSs = nki_series[filtered_top_DHSs]
        top_filtered_scores = filtered_scores[filtered_top_DHSs]
        
        comp_rec = list(comp_ser[top_rec_DHSs])
        chrom_rec = list(chrom_ser[top_rec_DHSs])
        start_rec = list(start_ser[top_rec_DHSs])
        end_rec = list(end_ser[top_rec_DHSs])
        numsamples_rec = list(numsamples_ser[top_rec_DHSs])
        identifier_rec = list(identifier_ser[top_rec_DHSs])
        
        count = 0
        total_count = 0
        return_biosample = []
        return_biosample.append(comp_rec)
        return_biosample.append(chrom_rec)
        return_biosample.append(start_rec)
        return_biosample.append(end_rec)
        return_biosample.append(numsamples_rec)
        return_biosample.append(list(top_rec_DHSs))
        return_biosample.append(list(top_filtered_scores))
        
        if comps_to_filter_out:
            return_biosample = filter_out_recs(return_biosample, comps_to_filter_out)
        
        return_list.append(return_biosample)
        
        #print results
        print('Biosample: %s (%s)' %(biosample_name, str(biosample)))
        
        if ROIurl == True:
            #creating ROIurl link
            if comps_to_filter:
                link_path = make_roi_url(biosample_name, str(biosample) + '_filtered', return_biosample)
            else:
                link_path = make_roi_url(biosample_name, biosample, return_biosample)
            print(link_path)
        
        if print_num > 0:
            print('   Number of Known DHSs: %s' %(len(known_positives)))
            print()
            print('   Recommended DHS Locations: ') 
            print('   Location | Component | Number of Samples | Biosample Signal/Average Signal | Left +DHS Signal | Right +DHS Signal')
        
        
        
            dhs_ids = top_rec_DHSs.to_numpy()
            #dhs to the right of recommended dhs
            dhs_right = dhs_ids[:]+1
            #dhs to the left of recommended dhs
            dhs_left = dhs_ids[:]-1
        
            #removing dhss that are "neighbored" across chromosomes
            chrm_list = list(np.unique(list(DHS_metadata['seqname'])))
            rows_per_chrm = [0]
            for chrm in chrm_list:
                max_row = max(rows_per_chrm)
                rows, col = DHS_metadata[DHS_metadata['seqname']==chrm].shape
                rows_per_chrm.append(rows + max_row)
            rows_per_chrm = np.array(rows_per_chrm)
        
            dhs_right = np.delete(dhs_right, np.isin(dhs_right, rows_per_chrm))
            dhs_left = np.delete(dhs_left, np.isin(dhs_left, rows_per_chrm-1))
        
            #determining presence/absense of dhss to right and left of recommended dhs
            right = dhs_array[biosample, dhs_right] 
            left = dhs_array[biosample, dhs_left]
        
            #signal data
            sample_rec_dhs_signal, rec_dhs_signal, rec_identifier = signal_values(list(top_rec_DHSs), signal_data)
            sample_right_dhs_signal, right_dhs_signal, right_identifier = signal_values(list(dhs_right), signal_data)
            sample_left_dhs_signal, left_dhs_signal, left_identifier = signal_values(list(dhs_left), signal_data)

        
        
            for i in range(print_num):
                chrom = chrom_rec[i]
                start = start_rec[i]
                end = end_rec[i]
                component = comp_rec[i]
                num_samples = numsamples_rec[i]
                sample_rec_dhs_signal_i = round(list(sample_rec_dhs_signal)[i], 3)
                rec_dhs_signal_i = round(float(list(rec_dhs_signal)[i]), 3)
                dhs_id = list(top_rec_DHSs)[i]
                right_id = list(np.where(dhs_right == dhs_id+1))[0] #+1
                if list(right_id) and right[right_id]==1:
                    sample_right_dhs_signal_i = round(sample_right_dhs_signal[int(right_id[0])], 3)
                    right_dhs_signal_i = round(right_dhs_signal[int(right_id[0])], 3)
                    right_print = str(round(sample_right_dhs_signal_i, 3)) + '/' + str(round(right_dhs_signal_i, 3))
                else:
                    right_print = 'N/A'
            
                left_id = list(np.where(dhs_left == dhs_id-1))[0]
                if list(left_id) and left[left_id]==1:
                    sample_left_dhs_signal_i = round(sample_left_dhs_signal[int(left_id[0])], 3)
                    left_dhs_signal_i = round(left_dhs_signal[int(left_id[0])], 3)
                    left_print = str(round(sample_left_dhs_signal_i, 3)) + '/' + str(round(left_dhs_signal_i, 3))
                else:
                    left_print = 'N/A'
                
                
                print('      %s:%s-%s | %s | %s | %s/%s | %s | %s'
                      %(chrom, start, end, component, num_samples, sample_rec_dhs_signal_i, rec_dhs_signal_i,
                       left_print, right_print))

    return return_list

#takes in recommendation info for a single biosample and the component to calculate precision values for.
#Returns precision values for this component
def component_precision_multiple_k(rec_info, target_comp, k_vals):
    comps = np.array(rec_info[0])
#     print(comps)
    comps_np = np.where(comps == target_comp, 1, 0)
#     print(comps_np)
    total_of_comp = np.sum(comps_np)
#     print(total_of_comp)
    count_list = []
    for k in k_vals:
        comp_k = comps_np[:k]
        count = np.sum(comp_k)
        count_list.append(count)
    counts = np.array(count_list)
    precision = counts/k_vals
    recall = counts/total_of_comp
    return precision, recall

#creates ROIurl links for recommended DHSs
def make_roi_url(biosample_name, biosample, rec_info):
    today = date.today()
    bed_path_name = os.path.join('/home/ctrader/public_html/', 'NMF_' + str(biosample_name) + '_' + str(biosample) + '_recs_CT' + today.strftime("%Y%m%d") + '.bed')
    f= open(bed_path_name, 'w')
    for i in range(100):
        f.write(rec_info[1][i] + '\t' + str(rec_info[2][i]) + '\t' + str(rec_info[3][i]) + '\t' + rec_info[0][i] + '\n')
    f.close()
    link_path = 'https://index.altius.org:3000/?application=viewer&roiURL=https%3A%2F%2Fencode%3Acollabor8%40resources.altius.org%2F~ctrader%2F' + 'NMF_' + str(biosample_name) + '_' + str(biosample) + '_recs_CT' + today.strftime("%Y%m%d") + '.bed'
    return link_path

def filter_in_recs(rec_info, comps_to_filter_in):
    rec_comps = np.array(rec_info[0])
    filter_inds = []
    for comp in comps_to_filter_in:
        filter_inds.extend(list(np.where(rec_comps == comp))[0])
    leftover_ind = np.array(filter_inds)
    return_comps = list(rec_comps[leftover_ind])
    return_chrm = list(np.array(rec_info[1])[leftover_ind])
    return_start = list(np.array(rec_info[2])[leftover_ind])
    return_end = list(np.array(rec_info[3])[leftover_ind])
    return_num_samples = list(np.array(rec_info[4])[leftover_ind])
    return_top_rec_DHSs = list(np.array(rec_info[5])[leftover_ind])
    return_top_filtered_scores = list(np.array(rec_info[6])[leftover_ind])
    
    return [return_comps, return_chrm, return_start, return_end, return_num_samples, return_top_rec_DHSs, return_top_filtered_scores, list(leftover_ind)]
# filter_test = filter_in_recs(unseen_pc_rec_info[0], ['Tissue invariant'])

def filter_out_recs(rec_info, comps_to_filter_out):
    rec_comps = np.array(rec_info[0])
    filter_inds = []
    for comp in comps_to_filter_out:
        filter_inds.extend(list(np.where(rec_comps == comp))[0])
    filter_inds = set(filter_inds)
    leftover_ind = np.array(list(set(range(len(rec_comps))) - filter_inds))
    return_comps = list(rec_comps[leftover_ind])
    return_chrm = list(np.array(rec_info[1])[leftover_ind])
    return_start = list(np.array(rec_info[2])[leftover_ind])
    return_end = list(np.array(rec_info[3])[leftover_ind])
    return_num_samples = list(np.array(rec_info[4])[leftover_ind])
    return_top_rec_DHSs = list(np.array(rec_info[5])[leftover_ind])
    return_top_filtered_scores = list(np.array(rec_info[6])[leftover_ind])
    
    return [return_comps, return_chrm, return_start, return_end, return_num_samples, return_top_rec_DHSs, return_top_filtered_scores, list(leftover_ind)]
# filter_test = filter_recs(unseen_rec_info[0], ['Tissue invariant'])

#Takes in recommended DHS indices and the signal data from a single biosample. Returns the average biosample signal
#across each recommended DHS and the average signal of all biosamples at each recommended DHS. 
def signal_values(dhs, signal_data):
    identifier_series = pd.Series(signal_data['name'])
    identifier = identifier_series[dhs]
    #Average biosample signal across each recommended DHS
    mean_signal_sample_series = pd.Series(signal_data['mean'])
    mean_signal_sample = mean_signal_sample_series[dhs]
    #Average signal of all biosamples at each recommended DHS
    mean_signal_DHS_series = pd.Series(DHS_metadata['mean_signal'])
    mean_signal_DHS = mean_signal_DHS_series[dhs]
    return list(mean_signal_sample), list(mean_signal_DHS), list(identifier)